# Dataset preparation for Pump-valves block data

***

**Author:** Juan M. Montes-Sánchez

**GitHub:** https://github.com/juamonsan

***

In this notebook you will find a step by step guide where you will:
- Import raw data captured with an ST device with custom firmware FP-SNS-DATALOG
- Preprocess the data for Neural Network training
- Convert the data to .csv or .wav files and store it as a sorted dataset

## Requirements

You **MUST** have:
- FP-SNS-DATALOG1 function pack from ST (please, download from https://www.st.com/en/embedded-software/fp-sns-datalog1.html)
- Raw data taken with FP-SNS-DATALOG1 on a STEVAL-MKSBOX1V1 (included in the repository)

Optional requirements include:
- STEVAL-MKSBOX1V1 SensorTile.box wireless multi sensor development kit. This is only needed for recording more data, in combination with FP-SNS-DATALOG1 (https://www.st.com/en/evaluation-tools/steval-mksbox1v1.html)

## Previous steps

### Importing modules for handling data

Then we import the python_SDK scripts (HSDatalog) from ST for data treating. It can be downloaded from: https://www.st.com/en/embedded-software/fp-sns-datalog1.html

For this work, the version used is v1.4.0 / 8-Jul-2022

The Python_SDK folder must be in the same directory where this notebook is located.

In [1]:
import sys

print("Python version {}.{}".format(sys.version_info.major, sys.version_info.minor))

sys.path.insert(1, './Python_SDK/')

from HSD.HSDatalog import HSDatalog
from HSD_utils.converters import HSDatalogConverter

Python version 3.7


---
### Other imports

In [2]:
# More imports
import matplotlib.pyplot as plt #for plotting raw data
import os #for system folders and files management
import pathlib #for system folders and files management
import pandas as pd #for Dataframe handling

from IPython.display import display # for folder selection dialog
from ipyfilechooser import FileChooser # for folder selection dialog

import numpy as np
import math
import csv

from IPython.display import Audio #for WAV generation

# End

In [3]:
# Make numpy values easier to read (optional)
np.set_printoptions(precision=3, suppress=True)

## Dataset loading and formating

### Initialization

The folder containing the raw data must be in the same directory where this notebook is located.

Initialize HSDatalog with raw data path (this code box shows a folder selection dialog):

In [4]:
# Create and display a FileChooser widget
fc = FileChooser('./')
fc.show_only_dirs=True
display(fc)

FileChooser(path='C:\Users\Usuario\Documents\GitHub\STEVAL-MKSBOX1V1_Dataset_creator', filename='', title='', …

In [24]:
if fc.selected_path==None:
    acq_folder = "./TEST_DATA/Raw_Data/STBOX_00001" #Default for not selection made. Change this path if raw data is located elsewhere
    print("No folder selected! Setting default path ({})".format(acq_folder))
else:
    acq_folder = fc.selected_path
hsd = HSDatalog(acquisition_folder=acq_folder)


---
<p><li> Listing of the available <b style="color:green">ACTIVE</b> sensors and the data files list:</li></p>

In [25]:
active_sensor_list = hsd.get_sensor_list(only_active=True)
label_list = hsd.get_acquisition_label_classes()

hsd.present_sensor_list(active_sensor_list)

file_names = hsd.get_dat_file_list()
print(file_names)

0) LIS3DHH
    0) ACC
2) LIS2DW12
    0) ACC
3) LIS2MDL
    0) MAG
4) LSM6DSOX
    0) ACC
    1) GYRO
6) MP23ABS1
    0) MIC
['LIS2DW12_ACC.dat', 'LIS2MDL_MAG.dat', 'LIS3DHH_ACC.dat', 'LSM6DSOX_ACC.dat', 'LSM6DSOX_GYRO.dat', 'MP23ABS1_MIC.dat']


---
### Pandas dataframes import
HSDatalog will be used to extract Pandas dataframes in a simple way. **MIC data will be excluded, if present.**

Starting from a DataFrame, by using the HSDatalogConverter module (already imported at *Previous steps* section), it is possible to convert the data of an acquisition in the following formats:
- CSV, TSV -> **CSV will be created for each non-microphone data processed with this notebook**
- NanoEdge format
- ST UNICO format
- WAV (Only for Microphones) -> **WAV will be created for MIC data if present**

In [26]:
#TODO Separar por etiquetas
#Create a Panda dataframe list for all active sensors and subsensors listed (Excluding MIC, if any)

print("Creating Panda Dataframe for all available sensors... please wait")

k=0
dataframe=list()
for i in range(len(active_sensor_list)):
    current_sensor=active_sensor_list[i] #sensor in active_sensor_list
    for j in range(len(active_sensor_list[i].sensor_descriptor.sub_sensor_descriptor)):
        current_subsensor=current_sensor.sensor_descriptor.sub_sensor_descriptor[j] #subsensor
        if current_subsensor.sensor_type != 'MIC': #exclude MIC
            current_odr = current_sensor.sensor_status.sub_sensor_status[j].odr #odr truncated towards 0
            dataframe.append(hsd.get_dataframe(current_sensor.name, current_subsensor.sensor_type, labeled = True))
            dataframe[k].attrs['name']=str("{}_{}_{}Hz".format(current_subsensor.sensor_type, current_sensor.name, int(current_odr)))
            dataframe[k].attrs['freq']=current_odr
            k=k+1
        else: #only if MIC
            print(current_sensor.name+" sensor is a MICROPHONE. Please note that audio data will not be processed in this batch.")

print ("A total of {} different sensor or subsensor data will be processed:".format(len(dataframe)))
for i in range (len(dataframe)):
    print("{}- {}".format(i+1,dataframe[i].attrs['name']))
print("Done!")

Creating Panda Dataframe for all available sensors... please wait
MP23ABS1 sensor is a MICROPHONE. Please note that audio data will not be processed in this batch.
A total of 5 different sensor or subsensor data will be processed:
1- ACC_LIS3DHH_1100Hz
2- ACC_LIS2DW12_1600Hz
3- MAG_LIS2MDL_100Hz
4- ACC_LSM6DSOX_6667Hz
5- GYRO_LSM6DSOX_6667Hz
Done!


In [27]:
# If successfully loaded, we can see sample data of the first sensor like this
df_num=0
print("Dataframe[{}] is: {}".format(df_num,dataframe[df_num].attrs['name']))
print(dataframe[df_num])

Dataframe[0] is: ACC_LIS3DHH_1100Hz
              Time   A_x [g]   A_y [g]   A_z [g]  BUBBLE  NORMAL   STOP
0         0.315320 -1.022960  0.052668  0.075544   False   False  False
1         0.316227 -1.018400  0.024168  0.050312   False   False  False
2         0.317134 -1.023188  0.043700  0.037088   False   False  False
3         0.318041 -0.998564  0.017176  0.058368   False   False  False
4         0.318947 -1.015968  0.003648  0.021128   False   False  False
...            ...       ...       ...       ...     ...     ...    ...
239995  217.987726 -1.038236  0.030248  0.065740   False   False  False
239996  217.988634 -1.002744 -0.000304  0.105260   False   False  False
239997  217.989541 -1.013612  0.037544  0.027740   False   False  False
239998  217.990448 -1.028812  0.034504  0.065740   False   False  False
239999  217.991355 -1.034208  0.003268  0.110960   False   False  False

[240000 rows x 7 columns]


Now we will read the column names in the places where the tags should be so we can determine which dataset are we working with. Currently, 3 tags (3 classes) are supported, no more, no less. If the dataset is not the provided bubble detection dataset or old vs new dataset, but it is using other different 3 tags for 3 class detection, this code will also work. Please, check tag order in that case.

In [28]:
if dataframe[0].columns.values[4] == 'BUBBLE':
    print("BUBBLE DATASET DETECTED")
    TAG1='STOP'
    TAG2='NORMAL'
    TAG3='BUBBLE'
        
elif dataframe[0].columns.values[4] == 'NEW':
    print("OLD vs NEW DATASET DETECTED")
    TAG1='STOP'
    TAG2='NEW'
    TAG3='OLD'
else:
    print("OTHER DATASET DETECTED, please check the result carefully.")
    TAG1=dataframe[0].columns.values[4]
    TAG1=dataframe[0].columns.values[5]
    TAG1=dataframe[0].columns.values[6]
    print("Using only 3 tags: {},{},{}.".format(TAG1, TAG2, TAG3))

BUBBLE DATASET DETECTED


### Dataframes pre-procesing
Now we will process the data so we achieve:
1. Merge tags into a single "Class" column and remove untagged data
2. Divide data into 0,5s overlapping windows
3. Store all in .csv format, in separate folders for each sensor

At the end of this proccess the data will be ready to be reimported from the .csv for Neural Networks training.

#### 1. Merge tags into a single "Class" column and remove untagged data
If ST's FP-SNS-DATALOG1 tags where used, we can use that information for having the correct class for each sample. 

In this code, 3 tags called NEW, OLD and STOP are considered. Those names correspond to software tags, and were chosen during data recording with STEVAL-MKSBOX1V1. If you are using your own recorded and labelled data, you will have to modify the code so your tags are considered instead.

In [29]:
# Creating a single class column from the tags.
# TODO: error proofing the algorithm: auto-read tag names, discard more than one tag active at the same time, etc

print("Merging tags into class columns...")

for i in range(len(dataframe)):
    # List of arguments (conditions)
    conditions = [
        (dataframe[i][TAG1]), # STOP active, state is STOP (1)
        (dataframe[i][TAG2]),  # NEW or NORMAL active, state is NEW or NORMAL (2)
        (dataframe[i][TAG3]),  # OLD or BUBBLE active, state is OLD or BUBBLE (3)
        ]

    # list of values
    values = ['1', '2', '3'] #"No class" will result in a 0 value, so we start from 1
    #values = ['1', '3'] #"No class" will result in a 0 value, so we start from 1

    # create a new column and use np.select to assign values to it using our lists as arguments
    dataframe[i]['Class'] = np.select(conditions, values)

    # Keeps only the tagged data (although this is not necessary if we are splitting by class later)
    dataframe[i] = dataframe[i][(dataframe[i].Class) > '0']
    
    # Since we don't need them anymore, we can now delete the tags columns
    dataframe[i].drop([TAG1, TAG2, TAG3], axis=1, inplace=True)
    #dataframe[i].drop(['STOP', 'OLD'], axis=1, inplace=True)
    print("------------------------------------")
    print("{} successfully merged. This is a sample of the result:".format(dataframe[i].attrs['name']))
    print("------------------------------------")
    print(dataframe[i].head(2))
    
print("------------------------------------")
print("Finished!")
print("------------------------------------")

Merging tags into class columns...
------------------------------------
ACC_LIS3DHH_1100Hz successfully merged. This is a sample of the result:
------------------------------------
          Time   A_x [g]   A_y [g]   A_z [g] Class
5872  5.640793 -1.006012  0.019836  0.044004     2
5873  5.641700 -1.011256  0.011248  0.052972     2
------------------------------------
ACC_LIS2DW12_1600Hz successfully merged. This is a sample of the result:
------------------------------------
          Time   A_x [g]  A_y [g]   A_z [g] Class
8511  5.640145  0.000390  0.09594  0.011700     2
8512  5.640771  0.000585  0.09516  0.009945     2
------------------------------------
MAG_LIS2MDL_100Hz successfully merged. This is a sample of the result:
------------------------------------
         Time  M_x [gauss]  M_y [gauss]  M_z [gauss] Class
533  5.643306       1.9785        1.599       0.3825     2
534  5.653306       1.9740        1.596       0.3810     2
------------------------------------
ACC_LSM6DS

#### 2. Divide data into 0,5s overlapping windows
We can extend the lenght of our datasets by overlapping time windows. For doing so, first we need to declare groups for each dataset:

In [30]:
# Split the data by frame series of the same class
Sname=list()
Sfreq=list()
for i in range(len(dataframe)):
    
    #We don't want to lose the associated attributes, so we store them in external lists with same index
    Sname.append(dataframe[i].attrs['name'])
    Sfreq.append(dataframe[i].attrs['freq'])
    
    # We create a new column which increments each time Class column value changes
    dataframe[i]['Group'] = dataframe[i]['Class'].ne(dataframe[i]['Class'].shift()).cumsum()

    # And then the data is gruped by that column
    # See https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html#splitting-an-object-into-groups
    # df_grouped = df.groupby(['Class','Group']) #This line, in our case, results in the same division, since Groups are unique
    dataframe[i] = dataframe[i].groupby('Group')
    
    # Shows the new dataframe with grouped windows
    print("------------------------------------")
    print(Sname[i])
    print("------------------------------------")
    print(dataframe[i].head(1))
    


------------------------------------
ACC_LIS3DHH_1100Hz
------------------------------------
              Time   A_x [g]   A_y [g]   A_z [g] Class  Group
5872      5.640793 -1.006012  0.019836  0.044004     2      1
31293    28.694978 -1.017944  0.025460  0.060344     1      2
59326    54.119583 -1.035804  0.042712  0.017936     2      3
143493  130.454139 -1.001984  0.019532  0.035340     3      4
182605  165.929934 -1.017488  0.026448  0.060724     1      5
208155  189.104775 -1.010572  0.006080  0.058368     3      6
------------------------------------
ACC_LIS2DW12_1600Hz
------------------------------------
              Time   A_x [g]   A_y [g]   A_z [g] Class  Group
8511      5.640145  0.000390  0.095940  0.011700     2      1
45349    28.694527  0.002535  0.097305  0.012285     1      2
85975    54.119346  0.000780  0.096330  0.012480     2      3
207949  130.453887  0.004485  0.095160  0.011115     3      4
264642  165.929929  0.002925  0.096720  0.012285     1      5
301674 

---
Now our "dataframe" has changed to a "GroupBy" type object, which is essentially a dataframe with a group attribute. We can use that to split the data by group, so the windows only have same class lectures. Also, we can use the frequency of the sample to make all the windows fixed to the same time lenght. In this example, we chose a 0.5 seconds lenght window, each one overlapping from the middle with the next one (0.25 seconds steps).

In [31]:
dfs_windowed=[[] for y in range(len(dataframe))] #we create a new list, same size as dataframe

for i in range(len(dataframe)):
    window_size = int(Sfreq[i]/2)   #0.5 seg windows
    half_window_size = int(window_size/2) #overlap by the middle
    print("------------------------------------")
    print("{} data windowing start...".format(Sname[i]))
    print("------------------------------------")
    for name, group in dataframe[i]:
        win_counter=0
        for j in range(0,len(group),half_window_size):
            #dfs_windowed[i][len(dfs_windowed[i]):]=dataframe[i].group.iloc[j:j+window_size] #a list containing the windows
            dfs_windowed[i].append(group.iloc[j:j+window_size]) #a list containing the windows
            dfs_windowed[i][-1].drop(['Group'], axis=1, inplace=True) #we do not need the group column anymore
            win_counter=win_counter+1
            
        while (len(dfs_windowed[i][-1]) < window_size):
            print("Last group lenght:{} < window size ({}) -> Deleted".format(len(dfs_windowed[i][-1]),window_size))
            dfs_windowed[i].pop(-1) #if the data left on the group is smaller than the window size, discard
            win_counter=win_counter-1
            
        print("Class group {} splitted in {} windows".format(name,win_counter))        
print("------------------------------------")
print("Done!")
print("------------------------------------")

------------------------------------
ACC_LIS3DHH_1100Hz data windowing start...
------------------------------------
Last group lenght:220 < window size (550) -> Deleted
Last group lenght:495 < window size (550) -> Deleted
Class group 1 splitted in 75 windows
Last group lenght:217 < window size (550) -> Deleted
Last group lenght:492 < window size (550) -> Deleted
Class group 2 splitted in 80 windows
Last group lenght:223 < window size (550) -> Deleted
Last group lenght:498 < window size (550) -> Deleted
Class group 3 splitted in 65 windows
Last group lenght:270 < window size (550) -> Deleted
Last group lenght:545 < window size (550) -> Deleted
Class group 4 splitted in 124 windows


C:\Users\Usuario\anaconda3\envs\CONDA_environment\lib\site-packages\pandas\core\frame.py:4315: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  errors=errors,


Last group lenght:250 < window size (550) -> Deleted
Last group lenght:525 < window size (550) -> Deleted
Class group 5 splitted in 67 windows
Last group lenght:131 < window size (550) -> Deleted
Last group lenght:406 < window size (550) -> Deleted
Class group 6 splitted in 110 windows
------------------------------------
ACC_LIS2DW12_1600Hz data windowing start...
------------------------------------
Last group lenght:207 < window size (800) -> Deleted
Last group lenght:607 < window size (800) -> Deleted
Class group 1 splitted in 75 windows
Last group lenght:196 < window size (800) -> Deleted
Last group lenght:596 < window size (800) -> Deleted
Class group 2 splitted in 80 windows
Last group lenght:228 < window size (800) -> Deleted
Last group lenght:628 < window size (800) -> Deleted
Class group 3 splitted in 65 windows
Last group lenght:219 < window size (800) -> Deleted
Last group lenght:619 < window size (800) -> Deleted
Class group 4 splitted in 124 windows
Last group lenght:266 

In [32]:
#code for checking the correct format of dfs_windowed

print("dfs_windowed: length: {}, type: {}".format(len(dfs_windowed), type(dfs_windowed)))
for i in range(len(dfs_windowed)):
    print("dfs_windowed[{}]: length: {}, type: {}".format(i,len(dfs_windowed[i]), type(dfs_windowed[i])))
    print("dfs_windowed[{}][-1]: length: {}, type: {}".format(i,len(dfs_windowed[i][-1]), type(dfs_windowed[i][-1])))

dfs_windowed: length: 5, type: <class 'list'>
dfs_windowed[0]: length: 521, type: <class 'list'>
dfs_windowed[0][-1]: length: 550, type: <class 'pandas.core.frame.DataFrame'>
dfs_windowed[1]: length: 521, type: <class 'list'>
dfs_windowed[1][-1]: length: 800, type: <class 'pandas.core.frame.DataFrame'>
dfs_windowed[2]: length: 521, type: <class 'list'>
dfs_windowed[2][-1]: length: 50, type: <class 'pandas.core.frame.DataFrame'>
dfs_windowed[3]: length: 517, type: <class 'list'>
dfs_windowed[3][-1]: length: 3333, type: <class 'pandas.core.frame.DataFrame'>
dfs_windowed[4]: length: 517, type: <class 'list'>
dfs_windowed[4][-1]: length: 3333, type: <class 'pandas.core.frame.DataFrame'>


Next step is for formatting all windows so all of them start at index 0 and time 0:

In [33]:
# Exact_period:
# Set to True if you want to set an exact period between samples based on theorical frequency
# Set to False if you want to keep the real time between samples measured during data logging (recommended)
Exact_period=False 

print("------------------------------------")
print("Example BEFORE restarting index and time:")
print("------------------------------------")
print(dfs_windowed[0][0].tail(3))


if Exact_period:
    # Restarts indexes and Time from 0 for all windows (new exact period)
    for i in range(len(dfs_windowed)):
        period=1/Sfreq[i] # period of the sample (inverse to frequency)
        for j in range(len(dfs_windowed[i])):
            # next line for reseting index
            dfs_windowed[i][j].reset_index(drop=True, inplace=True)
            # From here we reset Time
            dfs_windowed[i][j].reset_index(inplace=True) # we replicate the new 0 starting index as a column
            dfs_windowed[i][j].pop('Time') # deletion of current "Time" column
            dfs_windowed[i][j] = dfs_windowed[i][j].rename(columns = {'index':'Time'}) # new index column is now "Time"
            dfs_windowed[i][j]['Time']=dfs_windowed[i][j]['Time'].mul(period) # Each position is multiplied by period
            
else:
    # Restart indexes and Time from 0 for all windows (deleting the offset but with original real measured period)
    for i in range(len(dfs_windowed)):
        period=1/Sfreq[i] # period of the sample (inverse to frequency)
        for j in range(len(dfs_windowed[i])):
            # next line for reseting index
            dfs_windowed[i][j].reset_index(drop=True, inplace=True)
            # From here we reset Time
            offset=dfs_windowed[i][j].loc[0,'Time']
            dfs_windowed[i][j].loc[:,'Time']=dfs_windowed[i][j].loc[:,'Time']-offset #gives a warning but appears to work properly
        
        
print("------------------------------------")
print("Example AFTER restarting index and time:")
print("------------------------------------")
print(dfs_windowed[0][0].tail(3))

------------------------------------
Example BEFORE restarting index and time:
------------------------------------
          Time   A_x [g]   A_y [g]   A_z [g] Class
6419  6.136808 -1.023340  0.022876  0.063232     2
6420  6.137715 -1.027900  0.059964  0.069084     2
6421  6.138621 -1.019312 -0.002964  0.038760     2
------------------------------------
Example AFTER restarting index and time:
------------------------------------
         Time   A_x [g]   A_y [g]   A_z [g] Class
547  0.496015 -1.023340  0.022876  0.063232     2
548  0.496922 -1.027900  0.059964  0.069084     2
549  0.497829 -1.019312 -0.002964  0.038760     2


C:\Users\Usuario\anaconda3\envs\CONDA_environment\lib\site-packages\pandas\core\indexing.py:1676: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self._setitem_single_column(ilocs[0], value, pi)


---
#### 3. Store all in .csv format, in separate folders for each sensor

This algorithm creates a file tree containing all windows, in separate sensor folders. If "Class_folders_active" is set to <b style="color:green">true</b>, which is recommended, each sample will be in its class corresponding folder and "Class" column is deleted from all dataframes before CSV conversion.

In [22]:
#Select destination folder:

# Create and display a FileChooser widget
fc2 = FileChooser('./')
fc2.show_only_dirs=True
display(fc2)

FileChooser(path='C:\Users\Usuario\Documents\GitHub\STEVAL-MKSBOX1V1_Dataset_creator', filename='', title='', …

In [34]:
if fc2.selected_path==None:
    output_folder = "TEST_DATA\\expData\\STBOX_00001_Windows" #Default for not selection made. Change this path if raw data is located elsewhere
    print("No folder selected! Setting default path ({})".format(output_folder))
else:
    output_folder = fc2.selected_path
    
Class_folders_active=False # True for class folder sorting, false for all windows together
with_header=False # True if the first row of the .csv has to contain a header
drop_timestamp=True # True for storing only data, with no time column

if not os.path.exists(output_folder): 
    os.makedirs(output_folder)

for i in range(len(dfs_windowed)): 
    exported_sensor_folder = os.path.join(output_folder,Sname[i]) #set sensor directory
    if not os.path.exists(exported_sensor_folder): 
        os.makedirs(exported_sensor_folder)
    print("CSV conversion for sensor {} began.".format(Sname[i]))
    
    if Class_folders_active:
        for j in range(len(dfs_windowed[i])): 
            window_class = dfs_windowed[i][j].loc[0,'Class'] #Class will be in the name of the window .csv file
            class_folder = os.path.join(exported_sensor_folder,window_class)
            if not os.path.exists(class_folder): 
                os.makedirs(class_folder)

            window_path = os.path.join(class_folder,"{}.csv".format(j)) #class folder, number of window in the filename

            dfs_windowed[i][j].pop('Class') # 'Class' column will be deleted before importing to .csv
            if drop_timestamp:
                dfs_windowed[i][j].pop('Time') # 'Time' column deleted before importing to .csv if drop_timestamp=True
            dfs_windowed[i][j].to_csv(path_or_buf=window_path,header=with_header,index=False)
    else:
        for j in range(len(dfs_windowed[i])): 
            window_class = dfs_windowed[i][j].loc[0,'Class'] #Class will be in the name of the window .csv file
            dfs_windowed[i][j].pop('Class') # 'Class' column will be deleted before importing to .csv
            window_path = os.path.join(exported_sensor_folder,"{0:0>4}_{1:0>2}.csv".format(j,window_class)) #number of window and class in the filename
            if drop_timestamp:
                dfs_windowed[i][j].pop('Time') # 'Time' column deleted before importing to .csv if drop_timestamp=True
            dfs_windowed[i][j].to_csv(path_or_buf=window_path,header=with_header,index=False)        

print("All {} dataframes converted successfully to CSV format in {}".format(i+1,output_folder))

CSV conversion for sensor ACC_LIS3DHH_1100Hz began.
CSV conversion for sensor ACC_LIS2DW12_1600Hz began.
CSV conversion for sensor MAG_LIS2MDL_100Hz began.
CSV conversion for sensor ACC_LSM6DSOX_6667Hz began.
CSV conversion for sensor GYRO_LSM6DSOX_6667Hz began.
All 5 dataframes converted successfully to CSV format in C:\Users\Usuario\Documents\GitHub\STEVAL-MKSBOX1V1_Dataset_creator\DATA\BUBBLE_DATA\Bubble_CSV_test


---
### WAV conversion for microphone data

This code creates a single .wav file for the microphone data. It has yet to be improved so it creates those .wav files in the same way we did with the rest of the samples, with tagged windows of the same length.


In [ ]:
import numpy as np
from IPython.display import Audio

for i in range(len(active_sensor_list)):
    for j in range(len(active_sensor_list[i].sensor_descriptor.sub_sensor_descriptor)):
        if active_sensor_list[i].sensor_descriptor.sub_sensor_descriptor[j].sensor_type == 'MIC': #Solo se procesan los micrófonos
            sensor_name = active_sensor_list[i].name
            print("MIC data from {} sensor found.".format(sensor_name))
            sensor = hsd.get_sensor(sensor_name)
            odr = sensor.sensor_status.sub_sensor_status[0].odr
            print("ODR of the sample is: {}Hz".format(odr))
            wav_file_name = os.path.join(output_folder, "{}_MIC.wav".format(sensor.name))
            wav_file = HSDatalogConverter.wav_create(wav_file_name, odr)

            chunkSize = 10000000
            isLastChunk = False
            sample_offset = 0
            while isLastChunk == 0:
                data = hsd.get_data_and_timestamps(sensor.name, "MIC", sample_start = sample_offset, sample_end = sample_offset+chunkSize, raw_flag = True)[0]  
                pcm_data = data.astype(np.int16)
                if len(data) < chunkSize:
                    isLastChunk = 1
                else:
                    sample_offset = sample_offset + chunkSize
                HSDatalogConverter.wav_append(wav_file,pcm_data)
            HSDatalogConverter.wav_close(wav_file)
            Audio(wav_file_name)
            